# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load, inspect, and process the FAIR² dataset published at Frontiers using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema available at:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values using the metadata. This helps inform the record sets and columns available for extraction.

In [ ]:
# List record sets in the dataset
print("Available record sets and fields:")
for rs in metadata.record_sets:
    print(f"- RecordSet @id: {rs['@id']} (name: {rs.get('name','')})")
    print("  Fields/columns (with @id):")
    for field in rs.get('fields', []):
        print(f"    - {field['@id']}: {field.get('name','')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id` from the overview.

In [ ]:
# You may need to update these @id values if the dataset structure is updated.
# Discover record set IDs using the print-out in the previous cell.

# Collect record set @ids (for example - substitute below with the real IDs from the printout)
record_set_ids = [rs['@id'] for rs in metadata.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Sample columns in {record_set_id}: {df.columns.tolist()[:6]}")
        print(df.head(2))
    else:
        print(f"No records found for {record_set_id}.")

# For demonstration, choose the first non-empty record set for later steps
main_record_set_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rs_id
        break

if main_record_set_id:
    print(f"Chosen main record set for EDA: {main_record_set_id}")
    print("Columns:", dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We will now:
- Select a numeric field for analysis,
- Filter records (e.g., molecular weight > threshold),
- Normalize values, and
- Group data by a categorical variable (for example, a sample or accession id).

> **Note**: Replace field `@id`s below with values seen in the DataFrame printed above. E.g., 'cr:MW' could be molecular weight, and 'cr:accession' or 'cr:Sample' could be categorical.

In [ ]:
import numpy as np

if main_record_set_id is None:
    print("No main record set available for EDA.")
else:
    df = dataframes[main_record_set_id]
    available_columns = df.columns.tolist()
    print(f"Available columns: {available_columns}")

    # Guess a numeric field and grouping field from available columns (change as appropriate)
    # Example: look for 'cr:MW' for molecular weight, 'cr:accession' for grouping
    numeric_field = None
    group_field = None

    for col in available_columns:
        if 'MW' in col or 'weight' in col:
            numeric_field = col
        if ('sample' in col.lower()) or ('accession' in col.lower()):
            group_field = col
    
    # Fallbacks if none found
    if numeric_field is None:
        numeric_field = available_columns[0]
    if group_field is None:
        group_field = available_columns[1] if len(available_columns) > 1 else available_columns[0]
    
    print(f"Numeric field (@id): {numeric_field}")
    print(f"Group field (@id): {group_field}")

    # Convert to numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    threshold = df[numeric_field].quantile(0.85) # top 15% by molecular weight (example)

    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} @id > {threshold} (top 15%): {len(filtered_df)} records")
    print(filtered_df[[group_field, numeric_field]].head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean())
        / filtered_df[numeric_field].std()
    )
    print(f"Normalized field added: {numeric_field}_normalized")
    print(
        filtered_df[[group_field, numeric_field, f"{numeric_field}_normalized"]].head()
    )

    # Group by group_field and take the mean normalized value per group (if grouping makes sense)
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[f"{numeric_field}_normalized"].mean().reset_index()
        print(f"Mean normalized {numeric_field} by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize the data distribution (e.g., histogram of normalized molecular weights or counts per group).

> You can use matplotlib or seaborn for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and not filtered_df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=30, kde=True)
    plt.title(f"Distribution of normalized {numeric_field} (top records)")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.show()

    if group_field in filtered_df.columns:
        top_groups = filtered_df[group_field].value_counts().nlargest(10).index.tolist()
        plt.figure(figsize=(10,4))
        sns.boxplot(data=filtered_df[filtered_df[group_field].isin(top_groups)], x=group_field, y=f"{numeric_field}_normalized")
        plt.xticks(rotation=45)
        plt.title(f"Boxplot of normalized {numeric_field} by top {group_field} categories")
        plt.show()

## 6. Conclusion
- We successfully loaded and explored the FAIR² protein dataset using `mlcroissant` referencing all schema elements by their `@id` fields.
- Record sets, field structure, and values were examined, and data was filtered and normalized for analysis.
- This demonstrates a FAIR, standards-based approach for exploring science datasets.

> Continue with domain-specific analysis by selecting other fields and filters using their `@id`s.